# Assignment 7 — Image Morphology

Implements erosion, dilation, boundary extraction, and hole filling.

This notebook is intentionally modular. **Style 1** uses OpenCV operations; **Style 2** uses OpenCV only for reading and Matplotlib for display, while the image-processing logic is written with arrays, loops, queues, and formulas.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LAB_DIR = Path.cwd()

def show_images(images, titles, cmap="gray", figsize=(14, 4)):
    """Display a sequence of images consistently in a notebook."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    axes = np.atleast_1d(axes)
    for axis, image, title in zip(axes, images, titles):
        axis.imshow(image, cmap=cmap)
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()


## Style 1 — OpenCV built-in operations

In [ ]:
def morphology_cv(binary):
    square = np.ones((5, 5), np.uint8)
    eroded = cv2.erode(binary, square, iterations=1)
    dilated = cv2.dilate(binary, square, iterations=1)
    boundary = cv2.subtract(binary, eroded)
    return eroded, dilated, boundary

def fill_holes_cv(binary):
    flood = binary.copy()
    mask = np.zeros((binary.shape[0] + 2, binary.shape[1] + 2), np.uint8)
    cv2.floodFill(flood, mask, (0, 0), 1)
    return binary | (1 - flood)


## Style 2 — DSA/manual implementation

In [ ]:
def morph_manual(binary, structuring_element):
    pad = structuring_element.shape[0] // 2; padded = np.pad(binary, pad); eroded = np.zeros_like(binary); dilated = np.zeros_like(binary)
    locations = np.argwhere(structuring_element == 1)
    for r in range(binary.shape[0]):
        for c in range(binary.shape[1]):
            window = padded[r:r + 2 * pad + 1, c:c + 2 * pad + 1]
            values = [window[x, y] for x, y in locations]
            eroded[r, c] = int(all(values)); dilated[r, c] = int(any(values))
    return eroded, dilated, binary - eroded

def fill_holes_manual(binary):
    outside = np.zeros_like(binary); stack = [(0, 0)]; outside[0, 0] = 1
    while stack:
        r, c = stack.pop()
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if (0 <= nr < binary.shape[0] and 0 <= nc < binary.shape[1]
                    and binary[nr, nc] == 0 and outside[nr, nc] == 0):
                outside[nr, nc] = 1; stack.append((nr, nc))
    return (binary | (1 - outside)).astype(np.uint8)


## Demonstration

In [ ]:
image = cv2.imread(str(LAB_DIR / "alphabet.png"), cv2.IMREAD_GRAYSCALE)
if image is None: image = np.zeros((64, 64), np.uint8); image[16:48, 16:48] = 255; image[25:39, 25:39] = 0
binary = (image > 127).astype(np.uint8); se = np.ones((5, 5), np.uint8)
e1, d1, b1 = morphology_cv(binary); e2, d2, b2 = morph_manual(binary, se)
print("Erosion equality:", np.array_equal(e1, e2), "Dilation equality:", np.array_equal(d1, d2))
show_images([binary, e1, d1, b1], ["Binary", "Erosion", "Dilation", "Boundary"])
